In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Embedding, GlobalAveragePooling1D, LayerNormalization, Dropout, Add, Input
from tensorflow.keras.optimizers import Adam
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# CSV 파일 로드
dataframe = pd.read_csv('./dataset/sentiment_data.csv')

# 데이터와 라벨 추출
sentences = dataframe['sentence'].astype(str).tolist()
labels = dataframe['label'].tolist()
# 이진 분류에서 출력이 sigmoid 값이고 loss가 binary_crossentropy이기 때문에
# float32로 변환
y = np.array(labels, dtype=np.float32)

embedding_dim = 128
max_len = 10

tokenizer = tf.keras.preprocessing.text.Tokenizer(oov_token='<OOV>')
# 단어 사전을 만듦
tokenizer.fit_on_texts(sentences)
# 문장을 단어 ID 배열로 변환
sequences = tokenizer.texts_to_sequences(sentences)
word_index = tokenizer.word_index
# padding(0번) 예약 인덱스
vocab_size = len(word_index) + 1

# data.shape: (문장개수, 10)
data = tf.keras.preprocessing.sequence.pad_sequences(sequences, maxlen=max_len, padding='post')
# data.shape

# stratify: 라벨 비율을 학습/검증에 동일하게 유지
X_train, X_val, y_train, y_val = train_test_split(data, labels, test_size=0.2, random_state=2026, stratify=y)

In [ ]:
dataframe.head()

In [ ]:
# 포지셔널 인코딩
def get_positional_encoding(max_len, d_model):
    pos_enc = np.zeros((max_len, d_model), dtype=np.float32)
    for pos in range(max_len):
        for i in range(0, d_model, 2):
            # 짝수 차원은 sin, 홀수 차원은 cos을 넣는 방식(transformer 논문 방식)
            pos_enc[pos, i] = np.sin(pos / (10000 ** (2 * i / d_model)))
            if i + 1 < d_model:
                pos_enc[pos, i + 1] = np.cos(pos / (10000 ** (2 * (i + 1) / d_model)))
    return pos_enc
    
positional_encoding = get_positional_encoding(max_len, embedding_dim)

In [ ]:
# 멀티헤드 어텐션 레이어
class MultiHeadSelfAttentionalLayer(tf.keras.layers.Layer):
    def __init__(self, num_heads, key_dim, dropout_rate=0.0):
        super().__init__()
        self.mha = tf.keras.layers.MultiHeadAttention(
            num_heads = num_heads,
            key_dim = key_dim,
            dropout = dropout_rate
        )
        self.norm = LayerNormalization(epsilon=1e-6)

    def call(self, x, training=None):
        attn = self.mha(query=x, value=x, key=x, training=training)
        return self.norm(x + attn)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Embedding, GlobalAveragePooling1D, LayerNormalization, Dropout, Add, Input
from tensorflow.keras.optimizers import Adam
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import keras



class MultiHeadSelfAttentionalLayer(tf.keras.layers.Layer):
    def __init__(self, num_heads, key_dim, dropout_rate=0.0):
        super().__init__()
        self.mha = tf.keras.layers.MultiHeadAttention(
            num_heads = num_heads,
            key_dim = key_dim,
            dropout = dropout_rate
        )
        self.norm = LayerNormalization(epsilon=1e-6)

    '''
        padding(0) 토큰을 의미 없는 값으로 처리하기
        1. Enbedding( ..., mask_zero=True)
        2. Self-Attention과 pooling에서 padding 위치를 무시하도록 mask를 전달 <- 선택
    '''
    def call(self, x, padding_mask=None, training=None):
        attn_mask = None
        if padding_mask is not None:
            # MultiHeadAttention이 기대하는 형태로 확장
            # (batch, 1, seq) -> query 길이 방향으로 설정
            attn_mask = padding_mask[:, tf.newaxis, :]
        attn = self.mha(query=x, value=x, key=x, training=training, attention_mask=attn_mask)
        return self.norm(x + attn)

num_heads = 8
key_dim = embedding_dim // num_heads

if embedding_dim % num_heads != 0:
    raise ValueError('embedding_dim은 num_heads로 나누어 떨어져야 합니다.')

inputs = Input(shape=(max_len,), dtype=tf.int32)
# padding_mask = tf.not_equal(inputs, 0)
padding_mask = keras.ops.not_equal(inputs, 0)

x = Embedding(input_dim=vocab_size, output_dim=embedding_dim, mask_zero=True)(inputs)
x = x + positional_encoding

x = MultiHeadSelfAttentionalLayer(num_heads=num_heads, key_dim=key_dim, dropout_rate=0.1)(
    x, padding_mask=padding_mask
)

# Pooling도 padding mask를 반영
gap = GlobalAveragePooling1D()
x = gap(x, mask=padding_mask)

x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
outputs = Dense(1, activation='sigmoid')(x)

model = Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(X_train, np.array(y_train), epochs=10, batch_size=16, validation_data=(X_val, np.array(y_val)))

In [ ]:
sample_texts = ["I absolutely love this!", "I can't stand this product"]
sample_sequences = tokenizer.texts_to_sequences(sample_texts)
sample_data = tf.keras.preprocessing.sequence.pad_sequences(sample_sequences, maxlen=max_len, padding='post')
predictions = model.predict(sample_data)

for i, text in enumerate(sample_texts):
    print(f"Text: {text}")
    print(f"Prediction: {'Positive' if predictions[i] > 0.5 else 'Negative'}")

# 1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step
# Text: I absolutely love this!
# Prediction: Positive
# Text: I can't stand this product
# Prediction: Negative

In [ ]:
sample_texts = ["This is the best thing I’ve ever bought!", "I’m extremely happy with the results.", "The service was terrible and rude.", "I regret buying this, it’s a complete waste of money."]
sample_sequences = tokenizer.texts_to_sequences(sample_texts)
sample_data = tf.keras.preprocessing.sequence.pad_sequences(sample_sequences, maxlen=max_len, padding='post')
predictions = model.predict(sample_data)

for i, text in enumerate(sample_texts):
    print(f"Text: {text}")
    print(f"Prediction: {'Positive' if predictions[i] > 0.5 else 'Negative'}")
# 1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 200ms/step
# Text: This is the best thing I’ve ever bought!
# Prediction: Negative
# Text: I’m extremely happy with the results.
# Prediction: Positive
# Text: The service was terrible and rude.
# Prediction: Negative
# Text: I regret buying this, it’s a complete waste of money.
# Prediction: Negative
    